# ICT-12e — Valeur de l'information pour l'animat incarné : EVPI/EVSI cross-engine (#13569)

**Sous-série ICT** (trajectoires intégrées, Epic #4588). Notebook compagnon du module [`ict.voi`](ict/voi.py) (PR #13652, tranche 1/3 — interface canonique EVPI/EVSI). **Tranche 2/3** de la greffe #13569 : ce notebook montre que la valeur de l'information, calculée par `ict.voi`, **matche les valeurs canoniques** des trois exemplaires natifs du dépôt :

- **DecInfer-6** (Infer.NET, .NET) — bayésien déterministe
- **DecPyMC-5** (PyMC, Python) — bayésien + MCMC
- **ict.voi** (NumPy) — interface analytique close-form commune

## Pourquoi cette interface

L'étude de la cognition incarnée demande à l'animat de **décider** sous incertitude : il observe avant d'agir si l'observation vaut son coût. La *valeur de l'information* howardienne (Howard, 1966) mesure combien l'animat est prêt à payer pour observer. Les trois moteurs ci-dessus calculent la même grandeur, et l'interface `ict.voi` pose la **signature canonique** qui leur est commune :

`DecisionProblem(states, prior, actions, utility)` → `evpi()`, `evsi()`, `evsi_net()`, `observation_is_worthwhile()`, `animat_decision_summary()`

## Trois exercices, animat incarné non dégénéré

1. **EVPI parapluie canonique** (DecPyMC-5 §2) — EVPI=3.5, animat prend le parapluie sans observer.
2. **EVSI sismique forage** (DecInfer-6 + DecPyMC-5 §3) — EVSI=253k > coût 50k → l'animat observe avant de forer.
3. **Animat incarné sens proprioceptif imparfait** — sens 85/75 vs oracle 100%, discrimination mesurée ~11% EVPI. Le proprioceptif imparfait n'est ni le bruit pur (EVSI=0) ni l'oracle (EVSI=EVPI).

Chaque section compare la valeur `ict.voi` à la **référence canonique** du moteur natif (citée dans le notebook natif) — pas une re-exécution cross-kernel, qui sort du scope immédiat.

> *Note méthodologique* : le contrôle runtime DecInfer-6 (.NET) ↔ DecPyMC-5 (PyMC) dans un seul notebook Jupyter nécessiterait un dual-kernel ; le scope de cette tranche est de vérifier que l'interface `ict.voi` matche les valeurs canoniques déjà publiées par les notebooks natifs, et d'exercer la **capacité distinctive** de l'animat incarné (sens proprioceptif imparfait discriminant).


In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np

from ict.voi import (
    DecisionProblem,
    expected_utility_per_action,
    optimal_action_without_info,
    evpi,
    evsi,
    evsi_net,
    observation_is_worthwhile,
    animat_decision_summary,
)

print("module ict.voi chargé (tranche 2/3, #13569)")


module ict.voi chargé (tranche 2/3, #13569)


## Exercice 1 — EVPI parapluie canonique (DecPyMC-5 §2)

Scénario : un animat doit décider s'il prend son parapluie. Pluie (P=0.3), soleil (P=0.7). Utilités :

- (parapluie, pluie) = 0
- (parapluie, soleil) = -5
- (pas_parapluie, pluie) = -50
- (pas_parapluie, soleil) = 0

**EVPI attendu = 3.5** (cf DecPyMC-5 §2 verbatim). Sans observation, l'animat prend le parapluie (EU=-3.5 vs -15 pour pas_parapluie).


In [2]:
# Exercice 1 : EVPI parapluie canonique
pb_parapluie = DecisionProblem(
    states=("pluie", "soleil"),
    prior=(0.3, 0.7),
    actions=("parapluie", "pas_parapluie"),
    utility=[
        [0.0, -50.0],
        [-5.0, 0.0],
    ],
)

eu, action = optimal_action_without_info(pb_parapluie)
e = evpi(pb_parapluie)
print(f"EU par action (prior) : {expected_utility_per_action(pb_parapluie)}")
print(f"Best action sans info : {action} (EU={eu:.2f})")
print(f"EVPI parapluie (ict.voi) : {e:.3f}")
print(f"EVPI attendu (DecPyMC-5 §2) : 3.500")
assert abs(e - 3.5) < 1e-9, f"Mismatch : {e} vs 3.5"
print("OK : ict.voi EVPI == DecPyMC-5 §2 référence")


EU par action (prior) : [ -3.5 -15. ]
Best action sans info : parapluie (EU=-3.50)
EVPI parapluie (ict.voi) : 3.500
EVPI attendu (DecPyMC-5 §2) : 3.500
OK : ict.voi EVPI == DecPyMC-5 §2 référence


## Exercice 2 — EVSI sismique forage (DecInfer-6 + DecPyMC-5 §3)

Scénario : un animat doit décider s'il fore un puits de pétrole. P(pétrole)=0.3, P(pas_pétrole)=0.7.

Utilités :

- (forer, pétrole) = 1.5M (gain - coût forage)
- (forer, pas_pétrole) = -0.5M (-coût_forage)
- (vendre, *) = 0.2M (prix_vente terrain)

**Senseur sismique** : sensibilité 0.90, spécificité 0.80. Coût d'observation : 50k.

**EVSI attendu ≈ 253k**, **EVSI_net = 203k > 0** → l'animat observe avant de forer (cf DecInfer-6 + DecPyMC-5 §3).


In [3]:
# Exercice 2 : EVSI sismique forage
pb_forage = DecisionProblem(
    states=("petrole", "pas_petrole"),
    prior=(0.3, 0.7),
    actions=("forer", "vendre"),
    utility=[
        [1_500_000.0, 200_000.0],
        [-500_000.0, 200_000.0],
    ],
)

L_seismic = np.array([
    [0.90, 0.10],  # petrole -> [test+, test-]
    [0.20, 0.80],  # pas_petrole -> [test+, test-]
])

summary = animat_decision_summary(pb_forage, L_seismic, cost=50_000.0)
print(f"EU sans info : {summary['eu_no_info']:.0f}")
print(f"Best sans info : {summary['best_no_info']}")
print(f"EVPI forage : {summary['evpi']:.0f}")
print(f"EVSI sismique : {summary['evsi']:.0f}")
print(f"EVSI_net (cout 50k) : {summary['evsi_net']:.0f}")
print(f"Observe? : {summary['observe']}")

# Cross-check vs DecInfer-6 / DecPyMC-5 §3 (valeurs canoniques)
assert summary['evsi_net'] > 0, "EVSI_net doit etre > 0 (rentable)"
assert summary['observe'] is True
print("OK : ict.voi EVSI_net > 0, animat observe avant de forer (cf DecInfer-6 §3)")


EU sans info : 200000
Best sans info : vendre
EVPI forage : 390000
EVSI sismique : 253000
EVSI_net (cout 50k) : 203000
Observe? : True
OK : ict.voi EVSI_net > 0, animat observe avant de forer (cf DecInfer-6 §3)


## Exercice 3 — Animat incarné sens proprioceptif imparfait

L'**animat incarné** a un sens proprioceptif imparfait : pas un oracle (100% fiable) ni un senseur uniforme (0% informatif), mais un senseur réaliste avec **85% de sensibilité** et **75% de spécificité** sur le scénario parapluie.

Cette **imperfection mesurée** est la marque de l'incarnation : le senseur du corps n'est pas parfait. La discrimination mesurée doit être **nette** (≠ 0% senseur uniforme, ≠ 100% oracle) mais **bornée** (≠ 100% EVPI).

**Attendu** : EVSI proprioceptif ~ 10-15% de l'EVPI, EVSI < coût pour un coût d'observation trop élevé.


In [4]:
# Exercice 3 : Animat incarne sens proprioceptif imparfait
L_proprio = np.array([
    [0.85, 0.15],  # pluie -> 85% sens "+", 15% bruit
    [0.25, 0.75],  # soleil -> 25% faux "+", 75% correct "-"
])

e_evpi = evpi(pb_parapluie)
e_proprio = evsi(pb_parapluie, L_proprio)
ratio = e_proprio / e_evpi

print(f"EVPI parapluie : {e_evpi:.3f}")
print(f"EVSI proprioceptif (85/75) : {e_proprio:.3f}")
print(f"Ratio EVSI/EVPI : {ratio*100:.1f}%")
print()
print("Calibration animat incarne :")
print(f"  Senseur parfait (oracle) : EVSI = {evsi(pb_parapluie, np.eye(2)):.3f} = EVPI")
print(f"  Senseur uniforme (bruit) : EVSI = {evsi(pb_parapluie, np.ones((2,2))*0.5):.6f} = 0")
print(f"  Senseur proprioceptif 85/75 : EVSI = {e_proprio:.3f} ({ratio*100:.1f}% EVPI)")
print()
print("Discrimination mesuree : le proprioceptif est NETTEMENT distinct")
print(f"  du bruit (0%) et de l oracle (100%), a {ratio*100:.1f}% de l EVPI.")
assert 0.05 < ratio < 0.20, f"Ratio {ratio:.3f} hors plage attendue [5%, 20%]"
print("\nOK : discrimination nette (5%-20% EVPI), animat incarne distinct")

# Cout d observation : si trop eleve, l animat n observe pas
summary_cheap = animat_decision_summary(pb_parapluie, L_proprio, cost=0.1)
summary_expensive = animat_decision_summary(pb_parapluie, L_proprio, cost=0.5)
print(f"\nCout 0.1 : EVSI_net = {summary_cheap['evsi_net']:.3f}, observe? {summary_cheap['observe']}")
print(f"Cout 0.5 : EVSI_net = {summary_expensive['evsi_net']:.3f}, observe? {summary_expensive['observe']}")
print("\nLe proprioceptif est rentable quand le cout d observation reste sous EVSI.")


EVPI parapluie : 3.500
EVSI proprioceptif (85/75) : 0.375
Ratio EVSI/EVPI : 10.7%

Calibration animat incarne :
  Senseur parfait (oracle) : EVSI = 3.500 = EVPI
  Senseur uniforme (bruit) : EVSI = 0.000000 = 0
  Senseur proprioceptif 85/75 : EVSI = 0.375 (10.7% EVPI)

Discrimination mesuree : le proprioceptif est NETTEMENT distinct
  du bruit (0%) et de l oracle (100%), a 10.7% de l EVPI.

OK : discrimination nette (5%-20% EVPI), animat incarne distinct

Cout 0.1 : EVSI_net = 0.275, observe? True
Cout 0.5 : EVSI_net = -0.125, observe? False

Le proprioceptif est rentable quand le cout d observation reste sous EVSI.


## Conclusion — EVPI/EVSI cross-engine pour l'animat incarné

Ce notebook montre que `ict.voi` (interface analytique commune, tranche 1/3 #13569) reproduit les **valeurs canoniques** des trois exemplaires natifs du dépôt :

- **EVPI parapluie** = 3.5 (DecPyMC-5 §2) ✓
- **EVSI sismique forage** = 253k, EVSI_net > 0, animat observe (DecInfer-6 + DecPyMC-5 §3) ✓
- **Animat incarné sens proprioceptif** = 10.7% EVPI, discrimination nette (≠ 0% bruit, ≠ 100% oracle) ✓

### Ce qui distingue cette tranche 2/3

**L'interface est unifiée, pas dupliquée.** Les trois exercices accèdent à EVPI/EVSI via le même appel canonique `ict.voi.animat_decision_summary()` (ou ses composants `evpi`, `evsi`). Les notebooks natifs DecInfer-6 et DecPyMC-5 restent les organes de calcul ; `ict.voi` en est la **cible de test commune**.

**L'animat incarné est non dégénéré.** Le sens proprioceptif 85/75 discrimine (10.7% EVPI, ≠ 0% et ≠ 100%) sans tomber dans le cas trivial du senseur parfait ou du bruit pur — c'est la marque de l'incarnation, et le lieu où la valeur howardienne a un sens.

### Suite

- **Tranche 3/3** : extension MCMC (DecPyMC-5 §10) — variance de l'EVSI sous meta-prior.
- **Contrôle cross-kernel runtime** DecInfer-6 (.NET) ↔ PyMC : hors scope de cette tranche (dual-kernel notebook Jupyter complexe), pourrait faire l'objet d'un subagent `.NET` ou d'un notebook dual.

See #13569 (tranche 2/3 — interface analytique + notebook animat incarné).
